# 🛸 CNN pour la Classification de Drones RF — 


1. **`ImageFolder` remplacé** par un Dataset personnalisé qui lit la classe depuis le nom du fichier (BUI)
2. **`fc2 = nn.Linear(256, 13)`** — 13 classes réelles au lieu de 15
3. **Data leakage corrigé** — le split train/test se fait par enregistrement source, pas par segment
4. **Matrice de confusion corrigée** — 13 classes avec les vrais noms au lieu de 4

---
## Architecture des 13 classes (dataset DroneRF)
```
BUI 00000 → Classe  0 : Background
BUI 10000 → Classe  1 : Bebop Mode1 (Connected)
BUI 10001 → Classe  2 : Bebop Mode2 (Hovering)
BUI 10010 → Classe  3 : Bebop Mode3 (Flying)
BUI 10011 → Classe  4 : Bebop Mode4 (Flying + Video)
BUI 10100 → Classe  5 : AR Drone Mode1
BUI 10101 → Classe  6 : AR Drone Mode2
BUI 10110 → Classe  7 : AR Drone Mode3
BUI 10111 → Classe  8 : AR Drone Mode4
BUI 11000 → Classe  9 : Phantom Mode1
BUI 11001 → Classe 10 : Phantom Mode2
BUI 11010 → Classe 11 : Phantom Mode3
BUI 11011 → Classe 12 : Phantom Mode4
```
---

## 📦 PARTIE 1 — Imports et Configuration

In [ ]:
# ════════════════════════════════════════════════════════
# IMPORTS GÉNÉRAUX
# ════════════════════════════════════════════════════════

import torch                               # Framework de deep learning
import torch.nn as nn                      # Modules pour construire le réseau (couches, loss...)
import torch.nn.functional as F            # Fonctions : relu, softmax...
import torch.optim as optim                # Algorithmes d'optimisation : Adam, SGD...
from torch.utils.data import Dataset, DataLoader, random_split
# Dataset     : classe de base pour créer un dataset personnalisé
# DataLoader  : charge les données par lots (batches)
# random_split: divise un dataset en train/test

from torchvision import transforms         # Transformations d'images (resize, normalize...)
from PIL import Image                      # Lecture des fichiers PNG

import os                                  # Interaction avec le système de fichiers
import re                                  # Expressions régulières (pour extraire le BUI du nom de fichier)
import random                             # Pour mélanger les enregistrements
from collections import Counter            # Comptage rapide par classe
from tqdm import tqdm                      # Barre de progression

import matplotlib.pyplot as plt            # Graphiques
import numpy as np                         # Calcul numérique

# ════════════════════════════════════════════════════════
# CONFIGURATION GPU
# ════════════════════════════════════════════════════════

# Utilise le GPU si disponible, sinon le CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Appareil utilisé : {device}")
if device.type == 'cuda':
    # benchmark=True : PyTorch choisit l'algorithme de convolution le plus rapide
    # À utiliser uniquement quand la taille des images est fixe (64x64 ici)
    torch.backends.cudnn.benchmark = True
    print(f"   GPU : {torch.cuda.get_device_name(0)}")

✅ Appareil utilisé : cuda
   GPU : NVIDIA GeForce RTX 2050


---
## 🗂️ PARTIE 2 — Mapping des Classes et Dataset Personnalisé

### Pourquoi un Dataset personnalisé ?
 `ImageFolder`  nécessite des **sous-dossiers** pour chaque classe.
Ici les classes sont encodées dans le **nom du fichier** via le BUI binaire.
Exemple : `Phantom_M1_11000L_9_Seg94.png` → BUI `11000` → Classe 9 (Phantom Mode1)

In [ ]:
# ════════════════════════════════════════════════════════
# MAPPING BUI → NUMÉRO DE CLASSE
# ════════════════════════════════════════════════════════

# BUI = Binary Unique Identifier (identifiant binaire unique défini dans le dataset DroneRF)
# Chaque combinaison de drone + mode de vol a un BUI unique à 5 bits
BUI_TO_CLASS = {
    "00000": 0,    # Background (pas de drone)
    "10000": 1,    # Bebop — Mode 1 : Connected
    "10001": 2,    # Bebop — Mode 2 : Hovering
    "10010": 3,    # Bebop — Mode 3 : Flying
    "10011": 4,    # Bebop — Mode 4 : Flying + Video
    "10100": 5,    # AR Drone — Mode 1 : Connected
    "10101": 6,    # AR Drone — Mode 2 : Hovering
    "10110": 7,    # AR Drone — Mode 3 : Flying
    "10111": 8,    # AR Drone — Mode 4 : Flying + Video
    "11000": 9,    # Phantom — Mode 1 : Connected
    "11001": 10,   # Phantom — Mode 2 : Hovering
    "11010": 11,   # Phantom — Mode 3 : Flying
    "11011": 12,   # Phantom — Mode 4 : Flying + Video
}

# Noms lisibles pour la matrice de confusion
CLASS_NAMES = [
    "Background",
    "Bebop M1", "Bebop M2", "Bebop M3", "Bebop M4",
    "AR M1",    "AR M2",    "AR M3",    "AR M4",
    "Phantom M1","Phantom M2","Phantom M3","Phantom M4"
]

NB_CLASSES = len(BUI_TO_CLASS)  # = 13
print(f"Nombre de classes : {NB_CLASSES}")


# ════════════════════════════════════════════════════════
# FONCTION : EXTRAIRE LE BUI DEPUIS LE NOM DE FICHIER
# ════════════════════════════════════════════════════════

def extraire_bui(filename):
    """
    Extrait le BUI (5 chiffres binaires) depuis le nom du fichier.
    
    Exemples :
      'Phantom_M1_11000L_9_Seg94.png' → '11000'
      'Bebop_M2_10001H_3_Seg5.png'   → '10001'
      'AR_M3_10110L_7_Seg12.png'     → '10110'
    
    re.search(r'[01]{5}', filename) :
      [01]   = un caractère qui est soit 0 soit 1
      {5}    = exactement 5 fois
      → cherche la première séquence de 5 bits dans le nom
    """
    match = re.search(r'[01]{5}', filename)
    if match:
        return match.group()  # Retourne ex : '11000'
    return None               # Aucun BUI trouvé


# ════════════════════════════════════════════════════════
# FONCTION : EXTRAIRE L'ID D'ENREGISTREMENT
# ════════════════════════════════════════════════════════

def extraire_recording_id(filename):
    """
    Extrait un identifiant unique pour l'enregistrement SOURCE
    (sans le numéro de segment).
    
    But : éviter le data leakage dans le split train/test.
    Tous les segments d'un même enregistrement doivent être
    soit dans le train, soit dans le test — jamais les deux.
    
  
    
    Logique : on retire la partie '_SegX.png' à la fin
    """
    # re.sub remplace '_Seg' suivi de chiffres et '.png' par une chaîne vide
    recording_id = re.sub(r'_Seg\d+\.png$', '', filename, flags=re.IGNORECASE)
    return recording_id


# ════════════════════════════════════════════════════════
# DATASET PERSONNALISÉ
# ════════════════════════════════════════════════════════

class DroneRFDataset(Dataset):
    """
    Dataset personnalisé pour le projet DroneRF.
    
    Contrairement à ImageFolder, ce dataset :
    - Accepte des dossiers éparpillés (pas de racine commune)
    - Lit la classe depuis le BUI dans le nom du fichier
    - Gère les deux bandes (H et L) comme des exemples indépendants
    - Gère les segments multiples par enregistrement
    """

    def __init__(self, folders, transform=None):
        """
        folders   : liste de chemins vers les dossiers contenant les PNG
        transform : transformations à appliquer sur chaque image
        """
        self.transform = transform
        self.samples = []  # Liste de tuples : (chemin_image, numéro_classe)

        for folder in folders:
            # Vérifie que le dossier existe
            if not os.path.exists(folder):
                print(f"⚠️  Dossier introuvable : {folder}")
                continue

            # Parcourt tous les fichiers du dossier
            for filename in os.listdir(folder):
                if not filename.lower().endswith(".png"):
                    continue  # Ignore les fichiers qui ne sont pas des PNG

                # Extrait le BUI depuis le nom du fichier
                bui = extraire_bui(filename)

                if bui is None:
                    print(f"⚠️  BUI introuvable dans : {filename}")
                    continue

                if bui not in BUI_TO_CLASS:
                    print(f"⚠️  BUI inconnu '{bui}' dans : {filename}")
                    continue

                # Convertit le BUI en numéro de classe
                classe = BUI_TO_CLASS[bui]
                chemin = os.path.join(folder, filename)
                self.samples.append((chemin, classe))

        # Affiche un résumé du chargement
        comptage = Counter([c for _, c in self.samples])
        print(f"\n✅ {len(self.samples)} images chargées au total")
        print("\nRépartition par classe :")
        for i, nom in enumerate(CLASS_NAMES):
            print(f"   Classe {i:2d} | {nom:15s} | {comptage.get(i, 0):6d} images")

    def __len__(self):
        # Retourne le nombre total d'images
        return len(self.samples)

    def __getitem__(self, idx):
        # Retourne une image et sa classe pour un index donné
        chemin, classe = self.samples[idx]
        image = Image.open(chemin).convert("RGB")  # Charge l'image en RGB
        if self.transform:
            image = self.transform(image)           # Applique les transformations
        return image, classe

Nombre de classes : 13


---
## 📂 PARTIE 3 — Chargement des Données avec Split 

### Pourquoi un split par enregistrement et non par segment ?
Un enregistrement de 1 minute segmenté en 100 segments donne 100 images très similaires.
Si on mélange aléatoirement, le train peut avoir Seg0 à Seg80 et le test Seg81 à Seg99 **du même signal**.
Le modèle "voit" presque le même signal pendant l'entraînement → 100% de précision **artificielle**.
La solution : tous les segments d'un même enregistrement vont dans le même groupe (train OU test).

In [ ]:
# ════════════════════════════════════════════════════════
# TRANSFORMATIONS DES IMAGES
# ════════════════════════════════════════════════════════

data_transforms = transforms.Compose([
    transforms.Resize((64, 64)),      # Redimensionne en 64x64 pixels
    transforms.ToTensor(),            # Convertit PIL Image → tenseur [0,1]
    transforms.Normalize([0.5], [0.5]) # Normalise → valeurs entre [-1, +1]
                                       # Formule : (x - 0.5) / 0.5
])


# ════════════════════════════════════════════════════════
# CHEMINS VERS TES 4 DOSSIERS
# ════════════════════════════════════════════════════════

# ⚠️ AMAAANI ETTTT BALKIIIIIIIIIS  ICI MODIFIE CES CHEMINS selon l'emplacement réel de tes dossiers
# Chaque chemin pointe DIRECTEMENT vers le dossier contenant les PNG
FOLDERS = [
   r"C:\Users\user\Downloads\PPP\Backgroundactivities\Backgroundactivities",  # Bruit de fond (pas de drone)
   r"C:\Users\user\Downloads\PPP\Bebop_spectorgrams\Bebop",                    # Drone Bebop
   r"C:\Users\user\Downloads\PPP\data_processed\data_processed\spectrograms\ARDrone",  # Drone AR
   r"C:\Users\user\Downloads\PPP\Phantom_spectograms\Phantom"                  # Drone Phantom
]


# ════════════════════════════════════════════════════════
# CHARGEMENT DU DATASET COMPLET
# ════════════════════════════════════════════════════════

# On charge SANS transform d'abord pour faire le split correctement
# puis on appliquera les transforms après
full_dataset = DroneRFDataset(FOLDERS, transform=data_transforms)


# ════════════════════════════════════════════════════════
# SPLIT  : PAR ENREGISTREMENT (évite le data leakage)
# ════════════════════════════════════════════════════════

# Étape 1 : Grouper les indices par enregistrement source
# Un "enregistrement" = tous les segments d'un même fichier CSV original
# Exemple : Phantom_M1_11000L_9_Seg0 à Seg94 = 1 enregistrement

recording_to_indices = {}  # Dict : recording_id → liste d'indices dans le dataset

for idx, (chemin, classe) in enumerate(full_dataset.samples):
    filename = os.path.basename(chemin)           # Ex : Phantom_M1_11000L_9_Seg94.png
    rec_id = extraire_recording_id(filename)      # Ex : Phantom_M1_11000L_9
    
    if rec_id not in recording_to_indices:
        recording_to_indices[rec_id] = []
    recording_to_indices[rec_id].append(idx)

print(f"\n📊 Nombre d'enregistrements uniques : {len(recording_to_indices)}")
print(f"   Nombre moyen de segments par enregistrement : "
      f"{len(full_dataset) / len(recording_to_indices):.1f}")


# Étape 2 : Mélanger et diviser les ENREGISTREMENTS (90% train / 10% test)
all_recordings = list(recording_to_indices.keys())
random.seed(42)           # Seed fixe pour la reproductibilité
random.shuffle(all_recordings)  # Mélange aléatoire des enregistrements

split_point = int(0.9 * len(all_recordings))   # 90% des enregistrements pour le train
train_recordings = all_recordings[:split_point] # Enregistrements pour le train
test_recordings  = all_recordings[split_point:] # Enregistrements pour le test


# Étape 3 : Récupérer les indices de segments correspondants
train_indices = []
for rec in train_recordings:
    train_indices.extend(recording_to_indices[rec])  # Ajoute tous les segments de cet enregistrement

test_indices = []
for rec in test_recordings:
    test_indices.extend(recording_to_indices[rec])

print(f"\n✅ Split par enregistrement :")
print(f"   Enregistrements train : {len(train_recordings)} → {len(train_indices)} segments")
print(f"   Enregistrements test  : {len(test_recordings)}  → {len(test_indices)} segments")


# Étape 4 : Créer les sous-datasets avec les indices corrects
from torch.utils.data import Subset
# Subset : crée un sous-dataset à partir d'une liste d'indices
train_data = Subset(full_dataset, train_indices)
test_data  = Subset(full_dataset, test_indices)


# ════════════════════════════════════════════════════════
# CRÉATION DES DATALOADERS
# ════════════════════════════════════════════════════════

# DataLoader train :
# batch_size=256 : 256 images envoyées au GPU en même temps
# shuffle=True   : mélange les batches à chaque époque
# num_workers=4  : 4 processus parallèles pour charger les images
# pin_memory=True: accélère le transfert CPU → GPU
train_loader = DataLoader(
    train_data, 
    batch_size=64, 
    shuffle=True, 
    num_workers=0, 
    pin_memory=True
)

# DataLoader test :
# shuffle=False : ordre fixe pour la reproductibilité des résultats
test_loader = DataLoader(
    test_data, 
    batch_size=256, 
    shuffle=False
)

print(f"\n🚀 Pipeline prêt !")
print(f"   Total images     : {len(full_dataset)}")
print(f"   Images train     : {len(train_data)}")
print(f"   Images test      : {len(test_data)}")
print(f"   Batches par époque : {len(train_loader)}")


✅ 294377 images chargées au total

Répartition par classe :
   Classe  0 | Background      |  82000 images
   Classe  1 | Bebop M1        |  42000 images
   Classe  2 | Bebop M2        |  42000 images
   Classe  3 | Bebop M3        |  42000 images
   Classe  4 | Bebop M4        |  42000 images
   Classe  5 | AR M1           |   2377 images
   Classe  6 | AR M2           |      0 images
   Classe  7 | AR M3           |      0 images
   Classe  8 | AR M4           |      0 images
   Classe  9 | Phantom M1      |  42000 images
   Classe 10 | Phantom M2      |      0 images
   Classe 11 | Phantom M3      |      0 images
   Classe 12 | Phantom M4      |      0 images

📊 Nombre d'enregistrements uniques : 295
   Nombre moyen de segments par enregistrement : 997.9

✅ Split par enregistrement :
   Enregistrements train : 265 → 264377 segments
   Enregistrements test  : 30  → 30000 segments

🚀 Pipeline prêt !
   Total images     : 294377
   Images train     : 264377
   Images test      : 30000

---
## 🧠 PARTIE 4 — Architecture du Modèle CNN

In [ ]:
# ════════════════════════════════════════════════════════
# DÉFINITION DU MODÈLE CNN
# ════════════════════════════════════════════════════════

class DroneCNN(nn.Module):
    """
    Réseau de Neurones Convolutif pour la classification de drones RF.
    
    Architecture :
    Input (3x64x64)
        → Conv1 + ReLU + Pool  → (32x32x32)
        → Conv2 + ReLU + Pool  → (64x16x16)
        → Conv3 + ReLU + Pool  → (128x8x8)
        → Flatten              → (8192)
        → FC1 + ReLU + Dropout → (256)
        → FC2                  → (13 classes)
    """

    def __init__(self, nb_classes):
        """
        nb_classes : nombre de classes à prédire (13 dans notre cas)
                     Passé en paramètre pour éviter de coder en dur
        """
        super(DroneCNN, self).__init__()  # Initialise la classe parente nn.Module

        # ── ÉTAGE CONVOLUTIF 1 ──────────────────────────────────
        # Conv2d(canaux_entrée, canaux_sortie, taille_filtre, padding)
        # 3 canaux d'entrée (RGB), 32 filtres, filtre 3x3
        # padding=1 : conserve la taille spatiale après convolution
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)

        # MaxPool2d(taille, stride) : réduit la taille de moitié
        # Prend le maximum dans chaque zone 2x2
        # Partagé entre les 3 étages pour économiser la mémoire
        self.pool = nn.MaxPool2d(2, 2)

        # ── ÉTAGE CONVOLUTIF 2 ──────────────────────────────────
        # Entrée : 32 canaux (sortie conv1), Sortie : 64 filtres
        # Détecte des motifs plus complexes que conv1
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)

        # ── ÉTAGE CONVOLUTIF 3 ──────────────────────────────────
        # Entrée : 64 canaux, Sortie : 128 filtres
        # Détecte les structures spectrales de haut niveau
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)

        # ── COUCHES FULLY CONNECTED ──────────────────────────────
        # Après 3 poolings sur image 64x64 : 64 → 32 → 16 → 8
        # Volume final : 128 filtres × 8 × 8 = 8192 valeurs
        self.fc1 = nn.Linear(128 * 8 * 8, 256)  # 8192 → 256 neurones

        # ✅ CORRECTION : nb_classes=13 au lieu de 15
        # Le modèle a maintenant exactement autant de sorties que de classes réelles
        self.fc2 = nn.Linear(256, nb_classes)    # 256 → 13 scores (un par classe)

        # Dropout(p=0.5) : désactive aléatoirement 50% des neurones pendant l'entraînement
        # But : éviter l'overfitting (mémorisation des exemples d'entraînement)
        # Désactivé automatiquement en mode eval() (pendant l'évaluation)
        self.dropout = nn.Dropout(0.5)


    def forward(self, x):
        """
        Définit l'ordre de traversée des couches (propagation avant).
        x : batch d'images de forme (N, 3, 64, 64)
        """

        # Étage 1 : Convolution → Activation ReLU → MaxPooling
        # ReLU : met à 0 les valeurs négatives (non-linéarité)
        # Taille : (N, 3, 64, 64) → (N, 32, 32, 32)
        x = self.pool(F.relu(self.conv1(x)))

        # Étage 2 : idem
        # Taille : (N, 32, 32, 32) → (N, 64, 16, 16)
        x = self.pool(F.relu(self.conv2(x)))

        # Étage 3 : idem
        # Taille : (N, 64, 16, 16) → (N, 128, 8, 8)
        x = self.pool(F.relu(self.conv3(x)))

        # Aplatissement : transforme le tenseur 3D en vecteur 1D
        # (N, 128, 8, 8) → (N, 8192)
        # -1 : PyTorch calcule automatiquement cette dimension
        x = x.view(-1, 128 * 8 * 8)

        # Couche dense 1 : 8192 → 256 neurones + activation ReLU
        x = F.relu(self.fc1(x))

        # Dropout : désactive 50% des neurones (seulement en mode train)
        x = self.dropout(x)

        # Couche de sortie : 256 → 13 scores bruts (logits)
        # Pas de softmax ici : CrossEntropyLoss l'intègre en interne
        x = self.fc2(x)

        return x  # Forme : (N, 13)


# ════════════════════════════════════════════════════════
# INSTANCIATION DU MODÈLE
# ════════════════════════════════════════════════════════

# On passe NB_CLASSES en paramètre : plus besoin de modifier le code si les classes changent
model = DroneCNN(nb_classes=NB_CLASSES).to(device)

# Affiche un résumé de l'architecture
print("Architecture du modèle :")
print(model)

# Compte le nombre total de paramètres entraînables
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nNombre de paramètres : {total_params:,}")

Architecture du modèle :
DroneCNN(
  (conv1): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv3): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (fc1): Linear(in_features=8192, out_features=256, bias=True)
  (fc2): Linear(in_features=256, out_features=13, bias=True)
  (dropout): Dropout(p=0.5, inplace=False)
)

Nombre de paramètres : 2,193,997


---
## 🏋️ PARTIE 5 — Entraînement

In [ ]:
# ════════════════════════════════════════════════════════
# CONFIGURATION DE L'ENTRAÎNEMENT
# ════════════════════════════════════════════════════════

# S'assure que le modèle est sur le bon appareil (GPU ou CPU)
model.to(device)

# FONCTION DE PERTE :
# CrossEntropyLoss = standard pour la classification multi-classes
# Elle compare les 13 scores prédits avec la vraie classe
# et intègre un Softmax interne
criterion = nn.CrossEntropyLoss()

# OPTIMISEUR ADAM :
# Ajuste les poids du modèle pour minimiser la perte
# lr=0.001 : taux d'apprentissage standard (vitesse de modification des poids)
optimizer = optim.Adam(model.parameters(), lr=0.001)

# GRAD SCALER (Mixed Precision / AMP) :
# Utilise float16 au lieu de float32 sur le GPU → 2x plus rapide
# Uniquement disponible sur GPU NVIDIA
use_amp = device.type == 'cuda'  # Active AMP uniquement si GPU disponible
scaler = torch.amp.GradScaler('cuda') if use_amp else None

EPOCHS = 10  # Nombre de fois où le modèle voit tout le dataset

# Listes pour tracer les courbes d'apprentissage
historique_perte = []

print(f"🚀 Lancement de l'entraînement sur {device}")
print(f"   Époques     : {EPOCHS}")
print(f"   Batch size  : 256")
print(f"   Optimiseur  : Adam (lr=0.001)")
print(f"   Mixed Prec. : {'Activé' if use_amp else 'Désactivé (CPU)'}")
print()


# ════════════════════════════════════════════════════════
# BOUCLE D'ENTRAÎNEMENT
# ════════════════════════════════════════════════════════

for epoch in range(EPOCHS):

    # ── MODE ENTRAÎNEMENT ───────────────────────────────
    # Active le Dropout et les statistiques de BatchNorm
    model.train()
    running_loss = 0.0  # Accumule la perte de l'époque
    nb_batches = 0

    # tqdm : affiche une barre de progression avec infos en temps réel
    barre = tqdm(train_loader, desc=f"Époque {epoch+1}/{EPOCHS}")

    # ── BOUCLE SUR LES BATCHES ──────────────────────────
    for images, labels in barre:

        # Envoie les données sur le GPU (ou CPU)
        images = images.to(device)
        labels = labels.to(device)

        # Réinitialise les gradients du batch précédent
        # ⚠️ Obligatoire : sinon les gradients s'accumulent
        optimizer.zero_grad()

        if use_amp:
            # ── AVEC GPU : Précision Mixte (float16) ────
            with torch.amp.autocast('cuda'):
                outputs = model(images)           # Propagation avant
                loss = criterion(outputs, labels) # Calcul de la perte

            scaler.scale(loss).backward()  # Rétropropagation (calcule les gradients)
            scaler.step(optimizer)         # Met à jour les poids
            scaler.update()                # Ajuste le scaler pour le prochain batch
        else:
            # ── SANS GPU : Précision normale (float32) ──
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()    # Rétropropagation
            optimizer.step()   # Met à jour les poids

        # Accumule la perte
        running_loss += loss.item()  # .item() : tenseur → float Python
        nb_batches += 1

        # Met à jour l'affichage de la barre
        barre.set_postfix(perte=f"{loss.item():.4f}")

    # ── FIN D'ÉPOQUE ────────────────────────────────────

    perte_moyenne = running_loss / nb_batches
    historique_perte.append(perte_moyenne)

    # Sauvegarde les poids à chaque époque (checkpoint de sécurité)
    # state_dict() : dictionnaire de tous les poids et biais du modèle
    torch.save(model.state_dict(), f"drone_model_epoch_{epoch+1}.pth")
    print(f"✅ Époque {epoch+1}/{EPOCHS} — Perte moyenne : {perte_moyenne:.4f}")


print("\n🏆 Entraînement terminé !")


# ════════════════════════════════════════════════════════
# COURBE D'APPRENTISSAGE
# ════════════════════════════════════════════════════════

plt.figure(figsize=(8, 4))
plt.plot(range(1, EPOCHS+1), historique_perte, marker='o', color='steelblue')
plt.title("Courbe d'apprentissage — Perte par époque")
plt.xlabel("Époque")
plt.ylabel("Perte moyenne (CrossEntropy)")
plt.grid(True)
plt.tight_layout()
plt.show()
# Une bonne courbe descend progressivement vers 0
# Si elle stagne ou remonte → problème d'apprentissage

🚀 Lancement de l'entraînement sur cuda
   Époques     : 10
   Batch size  : 256
   Optimiseur  : Adam (lr=0.001)
   Mixed Prec. : Activé



Époque 1/10:  53%|█████▎    | 2188/4131 [22:14<19:45,  1.64it/s, perte=1.0003]     


KeyboardInterrupt: 

---
## 📊 PARTIE 6 — Évaluation et Matrice de Confusion

In [ ]:
# ════════════════════════════════════════════════════════
# ÉVALUATION SUR LE JEU DE TEST
# ════════════════════════════════════════════════════════

# model.eval() : passe en mode évaluation
# - Dropout DÉSACTIVÉ (tous les neurones actifs)
# - BatchNorm utilise les statistiques apprises
model.eval()

correct = 0  # Nombre de bonnes prédictions
total   = 0  # Nombre total d'images évaluées

# ✅ CORRECTION : utilise NB_CLASSES=13 au lieu de 4
# et CLASS_NAMES pour les vrais noms
conf_matrix = np.zeros((NB_CLASSES, NB_CLASSES), dtype=int)

print("🔍 Évaluation sur les images de test...")

# torch.no_grad() : désactive le calcul des gradients
# But : économiser la mémoire GPU (on n'entraîne pas ici)
with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="Évaluation"):

        images = images.to(device)
        labels = labels.to(device)

        # Propagation avant : produit 13 scores par image
        outputs = model(images)

        # torch.max(outputs, 1) : retourne (valeur_max, indice_max)
        # On garde seulement l'indice = la classe prédite
        _, predicted = torch.max(outputs, 1)

        total   += labels.size(0)                        # Nombre d'images dans ce batch
        correct += (predicted == labels).sum().item()    # Bonnes prédictions

        # Remplissage de la matrice de confusion
        # conf_matrix[vraie_classe, classe_predite] += 1
        for t, p in zip(labels.cpu().numpy(), predicted.cpu().numpy()):
            conf_matrix[t, p] += 1


# ════════════════════════════════════════════════════════
# MÉTRIQUES DE PERFORMANCE
# ════════════════════════════════════════════════════════

accuracy = 100 * correct / total
print(f"\n{'='*40}")
print(f"📊 PRÉCISION GLOBALE : {accuracy:.2f} %")
print(f"{'='*40}")

# Précision par classe (diagonale / total de la ligne)
print("\nPrécision par classe :")
for i, nom in enumerate(CLASS_NAMES):
    total_classe = conf_matrix[i].sum()
    if total_classe > 0:
        acc_classe = 100 * conf_matrix[i, i] / total_classe
        print(f"   {nom:15s} : {acc_classe:.1f}%  ({conf_matrix[i,i]}/{total_classe})")


# ════════════════════════════════════════════════════════
# MATRICE DE CONFUSION
# ════════════════════════════════════════════════════════

# ✅ CORRECTION : utilise les 13 vrais noms de classes
fig, ax = plt.subplots(figsize=(14, 12))

# imshow : affiche la matrice comme une carte de chaleur bleue
im = ax.imshow(conf_matrix, interpolation='nearest', cmap=plt.cm.Blues)
plt.colorbar(im, ax=ax)

ax.set_title("Matrice de Confusion — Modèle CNN DroneRF (13 classes)", 
             fontsize=14, pad=20)

# Étiquettes des axes avec les vrais noms
tick_marks = np.arange(NB_CLASSES)
ax.set_xticks(tick_marks)
ax.set_xticklabels(CLASS_NAMES, rotation=45, ha='right', fontsize=9)
ax.set_yticks(tick_marks)
ax.set_yticklabels(CLASS_NAMES, fontsize=9)

ax.set_ylabel('Vraie classe', fontsize=12)
ax.set_xlabel("Classe prédite par l'IA", fontsize=12)

# Affiche les chiffres dans chaque case
# Texte blanc si la case est foncée, noir sinon
thresh = conf_matrix.max() / 2.0
for i in range(NB_CLASSES):
    for j in range(NB_CLASSES):
        couleur = "white" if conf_matrix[i, j] > thresh else "black"
        ax.text(j, i, str(conf_matrix[i, j]),
                ha="center", va="center",
                color=couleur, fontsize=8)

plt.tight_layout()
plt.savefig("matrice_confusion_drone.png", dpi=150, bbox_inches='tight')
plt.show()
print("\n💾 Matrice sauvegardée : matrice_confusion_drone.png")

---
## 📝 Résumé des Corrections

| Problème original | Correction apportée |
|---|---|
| `ImageFolder` ne fonctionnait pas (pas de sous-dossiers) | `DroneRFDataset` personnalisé qui lit le BUI dans le nom du fichier |
| `fc2 = nn.Linear(256, 15)` — mauvais nombre de classes | `fc2 = nn.Linear(256, 13)` — 13 classes réelles |
| `random_split` → data leakage (segments du même enregistrement dans train ET test) | Split par enregistrement : tous les segments d'un même fichier source restent ensemble |
| `nb_classes = 4` dans l'évaluation — incomplet | `nb_classes = 13` avec les vrais noms de classes |
| Noms de classes inventés manuellement | Noms extraits automatiquement depuis `CLASS_NAMES` |

---
### ⚠️ Note sur le résultat attendu
Avec ces corrections, la précision ne sera **plus 100%**. C'était artificiel à cause du data leakage.
Un résultat réaliste pour ce type de dataset se situe entre **85% et 99%** selon le niveau de bruit (SNR).